# 14 — RAG Prompting

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## RAG Prompt Anatomy

In [3]:
CONTEXT = (
    "[Doc 1] LangChain is a framework for building LLM-powered applications. It provides abstractions for chains, agents, and memory.\n"
    "[Doc 2] LlamaIndex (formerly GPT Index) is a data framework for LLM applications with a focus on data ingestion and indexing.\n"
    "[Doc 3] ChromaDB is an open-source vector database designed for AI-native applications, enabling semantic search over embeddings."
)
question = "What is the difference between LangChain and LlamaIndex?"

rag_prompt = (
    "You are a helpful assistant. Answer the question using ONLY the provided context.\n"
    "If the answer is not in the context, say \'I don\'t have enough information.\'\n\n"
    f"Context:\n{CONTEXT}\n\n"
    f"Question: {question}"
)
result = chat([{"role":"user","content":rag_prompt}], temperature=0, max_tokens=200)
print("RAG Answer:")
print(result.strip())


RAG Answer:
I don't have enough information.


## Citation-Based RAG

In [4]:
rag_citation = (
    "Answer the question using the context.\n"
    "Cite your sources using [Doc N] notation at the end of each claim.\n\n"
    f"Context:\n{CONTEXT}\n\n"
    f"Question: {question}\n"
    "Answer with citations:"
)
result = chat([{"role":"user","content":rag_citation}], temperature=0, max_tokens=200)
print(result.strip())


Based on the provided context, the main difference between LangChain and LlamaIndex is their focus and functionality.

LangChain is a framework for building LLM-powered applications, providing abstractions for chains, agents, and memory [Doc 1]. This suggests that LangChain is more focused on the application development aspect, providing a structure for building and integrating LLMs into various applications.

On the other hand, LlamaIndex (formerly GPT Index) is a data framework for LLM applications, with a focus on data ingestion and indexing [Doc 2]. This implies that LlamaIndex is more focused on managing and organizing the data used by LLMs, rather than the application development itself.

In summary, while both frameworks are related to LLMs, LangChain is more focused on application development, and LlamaIndex is more focused on data management.

[Doc 1], [Doc 2]


## Handling Missing Context

In [5]:
unanswerable_q = "What is the pricing for LangChain Enterprise?"

strict_prompt = (
    "Answer ONLY from the context. If not found, reply exactly: Not found in context.\n\n"
    f"Context: {CONTEXT}\n\n"
    f"Question: {unanswerable_q}"
)
result = chat([{"role":"user","content":strict_prompt}], temperature=0, max_tokens=50)
print(f"Q: {unanswerable_q}")
print(f"A: {result.strip()}")


Q: What is the pricing for LangChain Enterprise?
A: Not found in context.


## Grounded RAG with Confidence

In [6]:
grounded_prompt = (
    "Answer the question based on the context below.\n"
    "After your answer, rate your confidence: HIGH (fully in context), MEDIUM (partial), LOW (inferred).\n\n"
    f"Context: {CONTEXT}\n\n"
    f"Question: {question}\n"
    "Answer:\nConfidence:"
)
result = chat([{"role":"user","content":grounded_prompt}], temperature=0, max_tokens=200)
print(result.strip())


Based on the context, LangChain is a framework for building LLM-powered applications, while LlamaIndex is a data framework for LLM applications with a focus on data ingestion and indexing. 

The main difference between LangChain and LlamaIndex appears to be their focus: LangChain focuses on building applications, whereas LlamaIndex focuses on data ingestion and indexing.

Confidence: MEDIUM (partial)


## Multi-Document Synthesis RAG

In [7]:
docs = [
    ("Doc A", "Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability."),
    ("Doc B", "Python 3.0 was released in 2008. It introduced major changes and is not backward-compatible with Python 2."),
    ("Doc C", "As of 2024, Python is the most popular programming language according to the TIOBE index."),
]

context = "\n".join(f"[{id}] {text}" for id, text in docs)
synthesis_q = "Summarize Python's history and current status based on the documents."

synthesis_prompt = (
    "Synthesize information from all documents to answer the question.\n"
    "Reference each document you use.\n\n"
    f"{context}\n\n"
    f"Question: {synthesis_q}\n"
    "Synthesis:"
)
result = chat([{"role":"user","content":synthesis_prompt}], temperature=0, max_tokens=200)
print(result.strip())


Based on the provided documents, here's a summary of Python's history and current status:

Python was created by Guido van Rossum and first released in 1991 (Doc A). It was designed to emphasize code readability. Over the years, significant changes were introduced with the release of Python 3.0 in 2008 (Doc B), which is not backward-compatible with Python 2. This indicates that the language has undergone substantial evolution since its inception. As of 2024, Python has become the most popular programming language according to the TIOBE index (Doc C), solidifying its position in the programming world.

References:
- Doc A: Information about Python's creation and initial release.
- Doc B: Details about the release of Python 3.0 and its impact on backward compatibility.
- Doc C: Current status of Python as the most popular programming language.
